# 02 · A/B Framework — a clean read on the View item -> Add to cart leak

**Analytical question.** The EDA walk identified `view_item -> add_to_cart` as the worst *proportional* leak in the funnel (80%). A realistic UI treatment on the product-detail page (PDP) — sticky CTA, price-per-stick copy, social proof — is expected to move this transition. Given a baseline CVR of ~20%, how many users per arm do we need to detect a +1pp absolute lift at 95%/80%? Once we simulate the experiment, what does a disciplined frequentist + Bayesian readout look like, and why does naive peeking invalidate the frequentist one?

**Why this matters.** An FMCG experimentation function that mis-sizes tests, peeks, or cannot explain expected loss to a brand team is a function that either ships losers or kills winners. This notebook is the end-to-end playbook: plan -> sanity-check -> read out -> decide -> translate to rupiah.

**Method summary.**
1. **Plan** — sample-size curves, pick MDE, fix horizon.
2. **Simulate** — seeded A/B with control 20.0%, treatment 21.0% over the planned horizon.
3. **Sanity** — Sample Ratio Mismatch (SRM) chi-square at alpha=0.01.
4. **Frequentist readout** — pooled-variance z-test, unpooled Wald CI on the lift.
5. **Bayesian readout** — Beta conjugate posteriors, P(T > C), expected loss, 95% credible interval.
6. **Peeking proof** — demonstrate empirically that a fixed-horizon test, peeked at 10 times, roughly doubles the Type-I rate. Motivates the always-valid Bayesian framing.
7. **Decision** — ship / hold / iterate rubric tied to both readouts.
8. **Business Impact** — rupiah at portfolio scale.

## 1 · Setup

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

from smokefreelab.analytics.viz import (
    COLOR_ACCENT,
    COLOR_GRID,
    COLOR_MUTED,
    COLOR_NEGATIVE,
    COLOR_POSITIVE,
    COLOR_PRIMARY,
    COLOR_TEXT,
    FONT_FAMILY,
    add_insight_annotation,
    apply_sfl_theme,
    format_rupiah,
)
from smokefreelab.experiment import (
    ArmStats,
    bayesian_test,
    check_srm,
    frequentist_test,
    sample_size_per_arm,
    simulate_peeking_inflation,
)

RNG = np.random.default_rng(2026)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 2 · Plan — sample size for a +1pp MDE

The EDA put `view_item -> add_to_cart` at **20.0%** baseline CVR. A PDP treatment typically moves this stage by 0.5-2pp; **we size for +1pp**, which is the smallest effect worth shipping (anything smaller is rounding error against seasonality and cohort drift). At alpha=0.05 two-sided and 80% power we need ~26K users per arm; at 90% power it jumps to ~35K — documented below so the scope-vs-power tradeoff is explicit.

In [2]:
BASELINE = 0.20
MDE_ABS = 0.01  # +1pp is the smallest effect worth shipping at portfolio scale.

planner_grid = [
    sample_size_per_arm(BASELINE, mde, alpha=0.05, power=p)
    for mde in (0.005, 0.010, 0.015, 0.020)
    for p in (0.80, 0.90)
]
planner_df = pd.DataFrame(
    [
        {
            "MDE (pp)": r.mde_abs * 100,
            "Power": r.power,
            "n per arm": r.sample_size_per_arm,
            "Total n": r.total_sample_size,
        }
        for r in planner_grid
    ]
)
planner_df

,MDE (pp),Power,n per arm,Total n
0,0.5000,0.8000,101400,202800
1,0.5000,0.9000,135746,271492
2,1.0000,0.8000,25580,51160
3,1.0000,0.9000,34244,68488
4,1.5000,0.8000,11469,22938
5,1.5000,0.9000,15354,30708
6,2.0000,0.8000,6507,13014
7,2.0000,0.9000,8711,17422


In [3]:
# Visual 1 — sample size curves across MDE x power.
mde_axis = np.linspace(0.003, 0.025, 50)
curves = {
    power: [sample_size_per_arm(BASELINE, mde, power=power).sample_size_per_arm for mde in mde_axis]
    for power in (0.80, 0.90, 0.95)
}
fig_power = go.Figure()
palette = {0.80: COLOR_PRIMARY, 0.90: COLOR_ACCENT, 0.95: COLOR_NEGATIVE}
for power, curve in curves.items():
    fig_power.add_trace(
        go.Scatter(
            x=mde_axis * 100,
            y=curve,
            mode="lines",
            name=f"Power = {power:.0%}",
            line={"color": palette[power], "width": 3},
            hovertemplate="MDE %{x:.2f}pp<br>n per arm: %{y:,}<extra></extra>",
        )
    )
fig_power.add_vline(
    x=MDE_ABS * 100,
    line={"color": COLOR_MUTED, "width": 1, "dash": "dot"},
    annotation_text="Chosen MDE = 1.0pp",
    annotation_position="top right",
    annotation_font={"color": COLOR_MUTED, "size": 11, "family": FONT_FAMILY},
)
fig_power.update_yaxes(title_text="Sample size per arm", type="log", tickformat=",")
fig_power.update_xaxes(title_text="Minimum Detectable Effect (pp)")
fig_power.update_layout(
    title="Sample size grows sharply as MDE shrinks",
    legend={"x": 0.75, "y": 0.95},
)
apply_sfl_theme(
    fig_power,
    height=420,
    subtitle="Baseline CVR 20.0%, alpha=0.05 two-sided. Chosen MDE = 1pp.",
)
add_insight_annotation(
    fig_power,
    text=(
        "<b>1pp at 80% power = ~26K per arm.</b><br>"
        "Halving the MDE to 0.5pp multiplies n by 4x."
    ),
    x=0.02, y=0.97,
    xref="paper", yref="paper",
    arrow=False,
)
fig_power.show()

In [4]:
plan = sample_size_per_arm(BASELINE, MDE_ABS, alpha=0.05, power=0.80)
print(f"Plan -> per arm: {plan.sample_size_per_arm:,} | total: {plan.total_sample_size:,}")
print(f"MDE = {plan.mde_abs * 100:.1f}pp | alpha = {plan.alpha:.2f} | power = {plan.power:.2f}")

Plan -> per arm: 25,580 | total: 51,160
MDE = 1.0pp | alpha = 0.05 | power = 0.80


## 3 · Simulate the experiment

We simulate a two-arm test at the planned horizon with **true control CVR 20.0%** and **true treatment CVR 21.0%** — the generative truth is exactly one MDE above baseline, so at 80% power we expect to reject H0 in ~80% of replicates. Assignment is 50/50 by a single random draw (deliberately imperfect so the SRM check has something to check).

In [5]:
TRUE_C = 0.200
TRUE_T = 0.210

n_each = plan.sample_size_per_arm
assignment = RNG.choice([0, 1], size=2 * n_each, replace=True)  # naive, may not land exactly 50/50
n_c_assigned = int((assignment == 0).sum())
n_t_assigned = int((assignment == 1).sum())

conv_c = int(RNG.binomial(n_c_assigned, TRUE_C))
conv_t = int(RNG.binomial(n_t_assigned, TRUE_T))

control = ArmStats("Control", n=n_c_assigned, conversions=conv_c)
treatment = ArmStats("Treatment", n=n_t_assigned, conversions=conv_t)

pd.DataFrame(
    {
        "Arm": [control.name, treatment.name],
        "n": [control.n, treatment.n],
        "Conversions": [control.conversions, treatment.conversions],
        "Observed CVR": [control.rate, treatment.rate],
    }
)

,Arm,n,Conversions,Observed CVR
0,Control,25631,5024,0.1960
1,Treatment,25529,5364,0.2101


## 4 · Sample Ratio Mismatch (SRM) — the gate before any readout

**Rule of the shop.** Never read the primary metric until SRM passes. A p-value below 0.01 usually means the randomisation logic is broken — a redirect that drops one arm, a sticky cookie bias, a filtered-down log. False alarms are expensive (you pause a clean test for nothing), true alarms save you from shipping a confounded result. The 0.01 threshold is the Microsoft / Airbnb industry default (Kohavi, Tang, Xu 2020).

In [6]:
srm = check_srm([control.n, treatment.n])
print(f"Observed: {srm.observed}  | Expected at 50/50: {srm.expected}")
print(f"chi2 = {srm.chi2:.4f} | p = {srm.p_value:.4f}")
print(f"SRM gate (alpha={srm.alpha}): {'PASS' if srm.passed else 'FAIL -- DO NOT READ OUT'}")

Observed: (25631, 25529)  | Expected at 50/50: (25580.0, 25580.0)
chi2 = 0.2034 | p = 0.6520
SRM gate (alpha=0.01): PASS


## 5 · Frequentist readout — z-test + unpooled Wald CI

The p-value uses the **pooled-variance** z-statistic (textbook H0: p_c = p_t). The CI on the absolute lift uses the **unpooled** Wald SE — standard practice since under H1 the two rates do not share a variance. Both numbers are returned so any auditor can re-derive without recomputing.

In [7]:
freq = frequentist_test(control, treatment, alpha=0.05)
pd.DataFrame(
    {
        "Metric": [
            "Control CVR",
            "Treatment CVR",
            "Absolute lift",
            "Relative lift",
            "z-statistic",
            "p-value (two-sided)",
            "95% CI on lift_abs",
            "Reject H0 at alpha=0.05?",
        ],
        "Value": [
            f"{freq.control.rate:.4f}",
            f"{freq.treatment.rate:.4f}",
            f"{freq.lift_abs * 100:+.3f}pp",
            f"{freq.lift_rel * 100:+.2f}%",
            f"{freq.z_stat:+.3f}",
            f"{freq.p_value:.4f}",
            f"[{freq.ci_low_abs * 100:+.3f}pp, {freq.ci_high_abs * 100:+.3f}pp]",
            "YES" if freq.significant else "NO",
        ],
    }
)

,Metric,Value
0,Control CVR,0.1960
1,Treatment CVR,0.2101
2,Absolute lift,+1.410pp
3,Relative lift,+7.19%
4,z-statistic,+3.964
5,p-value (two-sided),0.0001
6,95% CI on lift_abs,"[+0.713pp, +2.107pp]"
7,Reject H0 at alpha=0.05?,YES


In [8]:
# Visual 2 — forest-style lift CI against the zero line.
labels = ["Absolute lift (pp)"]
fig_ci = go.Figure()
fig_ci.add_trace(
    go.Scatter(
        x=[freq.lift_abs * 100],
        y=labels,
        mode="markers",
        marker={"color": COLOR_PRIMARY, "size": 14, "line": {"color": "white", "width": 2}},
        error_x={
            "type": "data",
            "symmetric": False,
            "array": [(freq.ci_high_abs - freq.lift_abs) * 100],
            "arrayminus": [(freq.lift_abs - freq.ci_low_abs) * 100],
            "color": COLOR_PRIMARY,
            "thickness": 2,
            "width": 12,
        },
        hovertemplate="Lift: %{x:+.3f}pp<extra></extra>",
        showlegend=False,
    )
)
fig_ci.add_vline(
    x=0.0,
    line={"color": COLOR_ACCENT, "width": 1.5, "dash": "dash"},
)
fig_ci.update_xaxes(title_text="Absolute lift (pp)", zeroline=False)
fig_ci.update_yaxes(showticklabels=True, title_text="")
fig_ci.update_layout(title="Frequentist 95% CI on the absolute lift")
apply_sfl_theme(
    fig_ci,
    height=240,
    subtitle=(
        f"z = {freq.z_stat:+.2f}, p = {freq.p_value:.4f}. "
        f"Reject H0 iff the interval excludes 0."
    ),
)
add_insight_annotation(
    fig_ci,
    text=(
        f"<b>{'Significant' if freq.significant else 'Not significant'} at alpha=0.05.</b><br>"
        f"CI: [{freq.ci_low_abs * 100:+.2f}pp, {freq.ci_high_abs * 100:+.2f}pp]"
    ),
    x=0.02, y=0.95,
    xref="paper", yref="paper",
    arrow=False,
)
fig_ci.show()

## 6 · Bayesian readout — posterior, P(T > C), expected loss

A `Beta(1, 1)` uniform prior is uninformative and recovers the MLE point estimate at the posterior mode — appropriate when we want the data to speak without bias toward a prior CVR. The posterior on the *difference* has no closed form, so we draw 200K samples from each marginal Beta and subtract. Three numbers matter:

- **P(T > C)** — the intuitive "how sure are we it is a winner."
- **95% credible interval on the absolute lift** — the direct Bayesian analogue of the frequentist CI.
- **Expected loss** — the posterior mean of the CVR we would forfeit if we ship the treatment and it is secretly worse. A hard ceiling of `loss <= 0.05pp` is a common bar to cross for a launch decision.

In [9]:
bayes = bayesian_test(control, treatment, n_draws=200_000, rng=np.random.default_rng(1))
lo, hi = bayes.credible_interval_abs
pd.DataFrame(
    {
        "Metric": [
            "P(Treatment > Control)",
            "95% credible interval on lift_abs",
            "Expected loss if we choose Treatment",
            "Expected loss if we choose Control",
            "Prior",
            "Posterior draws",
        ],
        "Value": [
            f"{bayes.prob_treatment_beats_control:.4f}",
            f"[{lo * 100:+.3f}pp, {hi * 100:+.3f}pp]",
            f"{bayes.expected_loss_choose_treatment * 100:.4f}pp",
            f"{bayes.expected_loss_choose_control * 100:.4f}pp",
            f"Beta({bayes.prior_alpha:g}, {bayes.prior_beta:g}) [uniform]",
            f"{bayes.n_draws:,}",
        ],
    }
)

,Metric,Value
0,P(Treatment > Control),1.0000
1,95% credible interval on lift_abs,"[+0.714pp, +2.109pp]"
2,Expected loss if we choose Treatment,0.0000pp
3,Expected loss if we choose Control,1.4119pp
4,Prior,"Beta(1, 1) [uniform]"
5,Posterior draws,"200,000"


In [10]:
# Visual 3 — posterior densities of the two CVRs, side-by-side.
x_grid = np.linspace(0.18, 0.24, 600)
post_c = stats.beta.pdf(
    x_grid,
    bayes.prior_alpha + control.conversions,
    bayes.prior_beta + control.n - control.conversions,
)
post_t = stats.beta.pdf(
    x_grid,
    bayes.prior_alpha + treatment.conversions,
    bayes.prior_beta + treatment.n - treatment.conversions,
)
fig_post = go.Figure()
fig_post.add_trace(
    go.Scatter(
        x=x_grid * 100,
        y=post_c,
        fill="tozeroy",
        name="Control posterior",
        line={"color": COLOR_PRIMARY, "width": 2.5},
        fillcolor="rgba(31, 58, 95, 0.35)",
        hovertemplate="CVR %{x:.2f}%<br>density %{y:.1f}<extra>Control</extra>",
    )
)
fig_post.add_trace(
    go.Scatter(
        x=x_grid * 100,
        y=post_t,
        fill="tozeroy",
        name="Treatment posterior",
        line={"color": COLOR_ACCENT, "width": 2.5},
        fillcolor="rgba(200, 85, 61, 0.35)",
        hovertemplate="CVR %{x:.2f}%<br>density %{y:.1f}<extra>Treatment</extra>",
    )
)
fig_post.update_xaxes(title_text="CVR (%)")
fig_post.update_yaxes(title_text="Posterior density", showticklabels=False)
fig_post.update_layout(title="Posterior distributions of the two CVRs")
apply_sfl_theme(
    fig_post,
    height=380,
    subtitle=(
        f"Beta({bayes.prior_alpha:g}, {bayes.prior_beta:g}) prior, "
        f"{bayes.n_draws:,} draws for difference posterior."
    ),
)
add_insight_annotation(
    fig_post,
    text=(
        f"<b>P(T > C) = {bayes.prob_treatment_beats_control:.3f}.</b><br>"
        f"95% CrI on lift: [{lo * 100:+.2f}pp, {hi * 100:+.2f}pp]"
    ),
    x=0.02, y=0.97,
    xref="paper", yref="paper",
    arrow=False,
)
fig_post.show()

In [11]:
# Visual 4 — posterior of the LIFT (Treatment - Control) with shaded CrI.
rng_plot = np.random.default_rng(7)
draws_c = rng_plot.beta(
    bayes.prior_alpha + control.conversions,
    bayes.prior_beta + control.n - control.conversions,
    size=200_000,
)
draws_t = rng_plot.beta(
    bayes.prior_alpha + treatment.conversions,
    bayes.prior_beta + treatment.n - treatment.conversions,
    size=200_000,
)
diff = (draws_t - draws_c) * 100
bins = np.linspace(diff.min(), diff.max(), 80)
hist, edges = np.histogram(diff, bins=bins, density=True)
centers = 0.5 * (edges[:-1] + edges[1:])

fig_diff = go.Figure()
fig_diff.add_trace(
    go.Bar(
        x=centers,
        y=hist,
        marker={
            "color": [COLOR_POSITIVE if c > 0 else COLOR_NEGATIVE for c in centers],
            "line": {"color": "white", "width": 0.5},
        },
        hovertemplate="Lift %{x:.2f}pp<br>density %{y:.1f}<extra></extra>",
        showlegend=False,
    )
)
fig_diff.add_vline(x=0.0, line={"color": COLOR_TEXT, "width": 1.5})
fig_diff.add_vline(
    x=lo * 100,
    line={"color": COLOR_MUTED, "width": 1, "dash": "dot"},
    annotation_text=f"{lo * 100:+.2f}pp",
    annotation_position="top",
    annotation_font={"color": COLOR_MUTED, "size": 10},
)
fig_diff.add_vline(
    x=hi * 100,
    line={"color": COLOR_MUTED, "width": 1, "dash": "dot"},
    annotation_text=f"{hi * 100:+.2f}pp",
    annotation_position="top",
    annotation_font={"color": COLOR_MUTED, "size": 10},
)
fig_diff.update_xaxes(title_text="Absolute lift, Treatment - Control (pp)")
fig_diff.update_yaxes(title_text="Posterior density", showticklabels=False)
fig_diff.update_layout(title="Posterior of the lift — mass above zero is the shipping signal")
apply_sfl_theme(
    fig_diff,
    height=380,
    subtitle=(
        f"Green mass to the right of zero = P(T > C) = "
        f"{bayes.prob_treatment_beats_control:.3f}."
    ),
)
add_insight_annotation(
    fig_diff,
    text=(
        f"<b>Expected loss if we ship T: {bayes.expected_loss_choose_treatment * 100:.3f}pp.</b><br>"
        "Tight loss is the launch criterion,<br>not just P(T > C)."
    ),
    x=0.02, y=0.97,
    xref="paper", yref="paper",
    arrow=False,
)
fig_diff.show()

## 7 · The peeking problem — why fixed-horizon frequentist tests cannot be monitored

A fixed-horizon frequentist test is only alpha-correct if you look *once*, at the horizon. Looking daily and stopping when `p < 0.05` sounds reasonable; it is not. We prove it empirically: 2,000 simulated **A/A** experiments (same true rate for both arms, so every rejection is a false positive), each peeked 1, 2, 5, 10, and 20 times at evenly-spaced interim points. The observed false-positive rate should stay at 5%; it does not.

**Two solutions in practice:**
1. **Bayesian posterior inference is always valid.** P(T > C) does not depend on horizon or stopping rule — it is the posterior probability given the data you happen to have. Monitor continuously, make a decision when expected loss and P(T > C) cross your thresholds.
2. **Sequential frequentist designs** (alpha-spending, mSPRT, always-valid confidence sequences) preserve Type-I under pre-declared peek schedules. Out of scope for this notebook — the Bayesian framing is cleaner for FMCG stakeholder communication.

In [12]:
peek_schedule = [1, 2, 5, 10, 20]
observed_alphas = {
    k: simulate_peeking_inflation(
        baseline_rate=BASELINE,
        n_total_per_arm=plan.sample_size_per_arm,
        n_peeks=k,
        alpha=0.05,
        n_sims=2_000,
        rng=np.random.default_rng(100 + k),
    )
    for k in peek_schedule
}
peek_df = pd.DataFrame(
    {
        "Peeks": peek_schedule,
        "Observed Type-I rate": [observed_alphas[k] for k in peek_schedule],
        "Inflation factor vs 5%": [observed_alphas[k] / 0.05 for k in peek_schedule],
    }
)
peek_df

,Peeks,Observed Type-I rate,Inflation factor vs 5%
0,1,0.0510,1.0200
1,2,0.0915,1.8300
2,5,0.1370,2.7400
3,10,0.2020,4.0400
4,20,0.2350,4.7000


In [13]:
# Visual 5 — peeking inflation bar chart.
fig_peek = go.Figure()
fig_peek.add_trace(
    go.Bar(
        x=[f"{k} peeks" for k in peek_schedule],
        y=[observed_alphas[k] * 100 for k in peek_schedule],
        marker={
            "color": [COLOR_POSITIVE if k == 1 else COLOR_NEGATIVE for k in peek_schedule],
            "line": {"color": "white", "width": 1.5},
        },
        text=[f"{observed_alphas[k] * 100:.1f}%" for k in peek_schedule],
        textposition="outside",
        textfont={"size": 13, "color": COLOR_TEXT},
        hovertemplate="%{x}<br>Observed alpha: %{y:.2f}%<extra></extra>",
        showlegend=False,
    )
)
fig_peek.add_hline(
    y=5.0,
    line={"color": COLOR_ACCENT, "width": 1.5, "dash": "dash"},
    annotation_text="Nominal alpha = 5%",
    annotation_position="top right",
    annotation_font={"color": COLOR_ACCENT, "size": 11},
)
fig_peek.update_yaxes(title_text="Observed Type-I rate (%)")
fig_peek.update_xaxes(title_text="Interim analyses per experiment")
fig_peek.update_layout(
    title="Peeking roughly doubles the false-positive rate by 10 looks",
)
apply_sfl_theme(
    fig_peek,
    height=420,
    subtitle="2,000 simulated A/A experiments per cell; baseline CVR 20.0%.",
)
add_insight_annotation(
    fig_peek,
    text=(
        "<b>Green = the one honest reading</b><br>"
        "at the pre-declared horizon.<br>"
        "Everything else is alpha-inflated."
    ),
    x=0.02, y=0.97,
    xref="paper", yref="paper",
    arrow=False,
)
fig_peek.show()

## 8 · Decision rubric — ship / hold / iterate

A single p-value is not a decision. An FMCG readout combines the frequentist CI (for auditability) with the Bayesian posterior (for expected-loss economics) and a pre-declared rubric:

| Signal | Ship (launch Treatment) | Hold (extend horizon) | Iterate (drop Treatment) |
|---|---|---|---|
| P(T > C) | >= 0.95 | 0.80 - 0.95 | < 0.80 |
| Expected loss choose T | <= 0.05pp | 0.05 - 0.20pp | > 0.20pp |
| CI lower bound | > 0.0pp | brackets 0 | < 0.0pp |
| SRM | PASS | PASS | any |

The rubric below maps the current experiment's three numbers to a decision automatically.

In [14]:
def classify_decision(
    p_t_beats_c: float,
    expected_loss_t: float,
    ci_low: float,
    srm_passed: bool,
) -> str:
    """Map three Bayesian/frequentist numbers to ship/hold/iterate."""
    if not srm_passed:
        return "BLOCKED (SRM fail)"
    if p_t_beats_c >= 0.95 and expected_loss_t <= 0.0005 and ci_low > 0:
        return "SHIP"
    if p_t_beats_c < 0.80 or ci_low < -0.005:
        return "ITERATE (drop this treatment)"
    return "HOLD (extend horizon or re-check for novelty/primacy effects)"


decision = classify_decision(
    bayes.prob_treatment_beats_control,
    bayes.expected_loss_choose_treatment,
    freq.ci_low_abs,
    srm.passed,
)
print("=" * 60)
print(f"DECISION: {decision}")
print("=" * 60)
print(f"P(T > C)              = {bayes.prob_treatment_beats_control:.4f}")
print(f"Expected loss choose T = {bayes.expected_loss_choose_treatment * 100:.4f}pp")
print(f"95% CI on lift_abs    = [{freq.ci_low_abs * 100:+.3f}pp, {freq.ci_high_abs * 100:+.3f}pp]")
print(f"SRM gate              = {'PASS' if srm.passed else 'FAIL'}")

DECISION: SHIP
P(T > C)              = 1.0000
Expected loss choose T = 0.0000pp
95% CI on lift_abs    = [+0.713pp, +2.107pp]
SRM gate              = PASS


## 9 · Caveats — what would make this readout wrong

1. **Novelty and primacy effects.** The first week of a UI change usually has unusual CVR — power users try the new thing, habitual users misclick. The horizon above ignores this; a production readout holds for a full week after ramp-up and reports the tail window separately.
2. **Simulated lift, not real lift.** We drew from `Binomial(n, 0.21)` to generate the treatment arm. The true lift is unknown; the same framework runs unchanged once real counts arrive.
3. **Single metric.** We tested add-to-cart rate. A treatment that lifts add-to-cart but tanks `purchase` (false lift on a shallow metric) would still show significant here. Production readouts always pair a proximate metric with a downstream guardrail.
4. **SRM on 50/50 only.** The check generalises to ramped rollouts by passing `expected_ratios`. If `traffic_source` distributions differ across arms, chi-square at arm level can still pass while a stratum-level SRM hides.
5. **Prior choice.** `Beta(1, 1)` is the honest default when we have no strong prior. A historical CVR of 20.0% with ~1000 users of observations would justify `Beta(200, 800)` — that would shrink the posterior toward 20.0% and require a larger effect to move `P(T > C)` above 0.95. The tradeoff is a conversation with the brand team, not a technical decision.
6. **Fixed-horizon only.** Peeking demonstrably inflates alpha. Any mid-experiment status update to stakeholders must surface `P(T > C)` and expected loss, not the interim p-value.

## 10 · Business Impact



**Framing.** A disciplined +1pp A/B win on the `view_item -> add_to_cart` stage compounds into the funnel wide the same way the funnel EDA sized. If the treatment ships, this single experiment is worth the following *annualised incremental CLV*, using a baseline of (250K monthly SFP registrants, IDR 450K LTV per activated user, 12 months):

In [15]:
# Re-use the funnel-EDA extrapolation — consistency across notebooks is non-negotiable.
MONTHLY_REGISTRANTS = 250_000
LTV_PER_USER = 450_000  # IDR
MONTHS_PER_YEAR = 12

# The EDA notebook established that a 1pp entry-to-purchase CVR lift is worth ~IDR 13.5B/yr.
# Because the view_item -> add_to_cart stage is one of five transitions, a 1pp lift at
# THIS stage propagates to a smaller entry-to-purchase lift. Approximation: the stage
# carries ~20% of the end-to-end conversion elasticity, so a 1pp stage lift is worth
# ~0.2pp end-to-end -> ~IDR 2.7B/yr. Precise decomposition is a follow-up deliverable
# (MMM-style attribution); we label this a lower bound.
stage_elasticity = 0.20  # fraction of end-to-end lift attributable to this stage
end_to_end_equivalent_pp = freq.lift_abs * stage_elasticity
annual_idr = (
    end_to_end_equivalent_pp
    * MONTHLY_REGISTRANTS
    * LTV_PER_USER
    * MONTHS_PER_YEAR
)

summary = pd.DataFrame(
    {
        "Quantity": [
            "Observed stage lift (Treatment - Control)",
            "End-to-end CVR equivalent (elasticity 0.20)",
            "Monthly SFP registrants",
            "LTV per activated user",
            "Annualised incremental CLV if shipped",
        ],
        "Value": [
            f"{freq.lift_abs * 100:+.3f}pp at view_item -> add_to_cart",
            f"{end_to_end_equivalent_pp * 100:+.3f}pp entry-to-purchase",
            f"{MONTHLY_REGISTRANTS:,}",
            f"IDR {LTV_PER_USER:,}",
            format_rupiah(annual_idr) + " / year",
        ],
    }
)
summary

,Quantity,Value
0,Observed stage lift (Treatment - Control),+1.410pp at view_item -> add_to_cart
1,End-to-end CVR equivalent (elasticity 0.20),+0.282pp entry-to-purchase
2,Monthly SFP registrants,"250,000"
3,LTV per activated user,"IDR 450,000"
4,Annualised incremental CLV if shipped,IDR 3.8B / year


**One-sentence takeaway for a hiring manager.** This framework sizes for a 1pp MDE, gates on SRM, reports both the two-proportion z-test CI and the Beta-Binomial posterior, proves empirically why peeking at a frequentist readout is wrong, and translates the lift to an annualised rupiah envelope consistent with the baseline above — end-to-end, in one reproducible notebook.

**A natural follow-up.** Wrap this framework in a Streamlit Experiment Designer so a PM can plug `(baseline, MDE, alpha, power)` in a sidebar and get the sample-size and readout tiles back. Same numbers, different medium.